# Estudiar los sismos

In [ ]:
import pandas as pd
import requests

url = (
    "https://sismosentido.sgc.gov.co/"
    "rest/resumenSismosConIntensidadBatch/-1"
)

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60
)
response.raise_for_status()

sismos = pd.DataFrame(response.json())
columnas = [
    "ID_SISMO", "FECHA", "LATITUD", "LONGITUD",
    "MAGNITUD", "PROFUNDIDAD", "I_MAX", "SITIO"
]
sismos = sismos[columnas]
sismos["FECHA"] = pd.to_datetime(
    sismos["FECHA"],
    format="%d/%m/%Y - %I:%M %p",
    errors="coerce"
)

print(f"Registros descargados: {len(sismos):,}")
sismos.head()

## Evolución diaria de los sismos (2016–2026)

La línea clara muestra el número exacto de sismos por día y la línea oscura muestra la media móvil de 30 días.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

inicio = pd.Timestamp("2016-01-01")
fin = pd.Timestamp("2026-12-31 23:59:59")

sismos_periodo = sismos.loc[
    sismos["FECHA"].between(inicio, fin)
].copy()

if sismos_periodo.empty:
    raise ValueError("No se encontraron sismos entre 2016 y 2026.")

# Terminar en el último día disponible para no representar fechas futuras como ceros.
ultimo_dia = min(fin.normalize(), sismos_periodo["FECHA"].max().normalize())
dias = pd.date_range(inicio, ultimo_dia, freq="D")

sismos_por_dia = (
    sismos_periodo.groupby(sismos_periodo["FECHA"].dt.normalize())
    .size()
    .reindex(dias, fill_value=0)
    .rename("NUMERO_SISMOS")
)

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(
    sismos_por_dia.index,
    sismos_por_dia.values,
    color="#023047",
    linewidth=0.7,
    alpha=0.65,
    label="Sismos por día"
)

ax.set_title("Evolución diaria de los sismos sentidos (2016–2026)", pad=14)
ax.set_xlabel("Fecha")
ax.set_ylabel("Número de sismos")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

sismos_por_dia.to_frame().tail()

## Magnitud máxima registrada por día (2016–2026)

Para cada fecha se conserva únicamente la mayor magnitud registrada. Los días sin sismos no tienen una magnitud máxima y no se representan como cero.

In [ ]:
magnitud_maxima_diaria = (
    sismos_periodo.assign(DIA=sismos_periodo["FECHA"].dt.normalize())
    .groupby("DIA")["MAGNITUD"]
    .max()
    .sort_index()
    .rename("MAGNITUD_MAXIMA")
)

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(
    magnitud_maxima_diaria.index,
    magnitud_maxima_diaria.values,
    color="#d62828",
    linewidth=0.8,
    alpha=0.8,
    label="Magnitud máxima diaria"
)

ax.set_title("Magnitud máxima registrada por día (2016–2026)", pad=14)
ax.set_xlabel("Fecha")
ax.set_ylabel("Magnitud máxima")
ax.set_xlim(inicio, ultimo_dia)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

magnitud_maxima_diaria.to_frame().tail()

In [ ]:
sismos = sismos.sort_values("MAGNITUD")

In [ ]:
sismos.tail()

In [ ]:
l = "Yumare, Venezuela"
l = l.split(", ")[1]
l

In [ ]:
sismos["PAIS"] = sismos["SITIO"].apply(lambda x: x.split(", ")[1])

In [1]:
import numpy as np

In [ ]:
Y = np.array([2,4,6,8,10,12])
X = np.array([
    [9, 8, 7],
    [6, 5, 4],
    [3, 2, 1]
])